# 🤖 本地 LLM + RAG 知識問答系統
## 基於 Ollama + ChromaDB + Streamlit，部署於 Google Colab

---

### 📖 技術架構
| 元件 | 工具 | 說明 |
|------|------|------|
| **嵌入模型** | `mxbai-embed-large` | 將文字轉為向量，用於語義搜尋 |
| **對話 LLM** | `ycchen/breeze-7b-instruct-v1_0` | 聯發科繁體中文優化語言模型 |
| **推論引擎** | Ollama | 本地 LLM 執行環境 |
| **向量資料庫** | ChromaDB | 儲存並查詢知識庫嵌入向量 |
| **Web UI** | Streamlit | 互動式問答介面 |
| **對外隧道** | pyngrok | 將 Colab 內部埠號公開為 HTTPS URL |

### ✅ 使用前請確認
1. **Runtime 已設為 T4 GPU**：選單 → `Runtime` → `Change runtime type` → `T4 GPU`
2. **已準備好 Ngrok authtoken**：前往 [https://ngrok.com](https://ngrok.com) 免費註冊後取得
3. **依序執行所有 Cell**（Cell 1 → Cell 9）

### ⏱️ 預計等待時間
- Cell 3（套件安裝）：約 2 分鐘
- Cell 4（Ollama 安裝）：約 1 分鐘
- Cell 5（模型下載）：⚠️ **約 5~15 分鐘**（模型總大小約 5GB）
- Cell 9（啟動服務）：約 30 秒

In [ ]:
# ============================================================
# ⚙️ Cell 2｜使用者設定區
# 只需修改此 Cell，其他 Cell 無需改動
# ============================================================

# ── [必填] Ngrok authtoken ──────────────────────────────────
# 前往 https://ngrok.com 免費註冊後，於 Dashboard 取得 authtoken
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

# ── [選填] Google Drive 整合設定 ────────────────────────────
# True  = 知識庫與向量索引持久保存於 Drive（推薦：重啟後無需重建索引）
# False = 使用記憶體模式，每次 Session 結束後資料消失
USE_GOOGLE_DRIVE = True

# Google Drive 中 QA Excel 檔案的路徑
# 若檔案不存在，將自動建立含 10 筆範例問答的知識庫
GDRIVE_QA_PATH = "/content/drive/MyDrive/RAG_QA/QA50.xlsx"

# ChromaDB 向量索引的持久化目錄（需 USE_GOOGLE_DRIVE = True）
CHROMA_PERSIST_DIR = "/content/drive/MyDrive/RAG_QA/chroma_db"

# ── [選填] AI 模型設定 ──────────────────────────────────────
# 文字嵌入模型（用於將文字轉為語義向量）
EMBEDDING_MODEL = "mxbai-embed-large"

# 對話語言模型（繁體中文優化版，聯發科 Breeze-7B）
LLM_MODEL = "ycchen/breeze-7b-instruct-v1_0"

# ── 設定驗證 ────────────────────────────────────────────────
assert NGROK_AUTHTOKEN != "YOUR_NGROK_AUTHTOKEN_HERE", \
    "❌ 請先在上方填入您的 Ngrok authtoken！"

print("✅ 設定讀取完成")
print(f"   📐 嵌入模型  : {EMBEDDING_MODEL}")
print(f"   💬 對話模型  : {LLM_MODEL}")
print(f"   💾 Google Drive : {'啟用' if USE_GOOGLE_DRIVE else '停用（記憶體模式）'}")

In [ ]:
# ============================================================
# 📦 Cell 3｜安裝系統套件
# ============================================================

print("📦 步驟 1/5：安裝系統工具...")

# 更新 apt 套件清單
!sudo apt-get update -qq 2>&1 | tail -3

# 安裝 pciutils（讓 Ollama 能正確偵測並使用 GPU）
# 安裝 curl（用於下載 Ollama 安裝腳本）
!sudo apt-get install -y -qq pciutils curl 2>&1 | tail -3

print("✅ 系統工具安裝完成")

In [ ]:
# ============================================================
# 🦙 Cell 4｜安裝並啟動 Ollama 背景服務
# ============================================================

import subprocess
import threading
import time
import requests

# ── 安裝 Ollama ─────────────────────────────────────────────
print("🦙 步驟 2/5：安裝 Ollama...")

install_result = subprocess.run(
    ["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
    capture_output=True,
    text=True
)

if install_result.returncode != 0:
    print(f"❌ Ollama 安裝失敗：\n{install_result.stderr}")
    raise RuntimeError("Ollama 安裝失敗，請檢查網路連線後重試")

print("✅ Ollama 安裝成功")

# ── 在背景執行緒中啟動 Ollama 服務 ─────────────────────────
def _start_ollama_server():
    """
    在獨立執行緒中啟動 Ollama 推論服務（daemon 模式）。
    使用 daemon=True 確保主程式結束時此執行緒也會自動終止。
    """
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,  # 隱藏標準輸出，避免刷屏
        stderr=subprocess.DEVNULL   # 隱藏錯誤輸出
    )

# 建立並啟動背景執行緒
server_thread = threading.Thread(target=_start_ollama_server, daemon=True)
server_thread.start()

# ── 輪詢等待 Ollama API 就緒（最多 30 秒）──────────────────
print("⏳ 等待 Ollama 服務就緒（最多 30 秒）...")
OLLAMA_API_URL = "http://localhost:11434/api/tags"

for elapsed in range(1, 31):
    try:
        resp = requests.get(OLLAMA_API_URL, timeout=2)
        if resp.status_code == 200:
            print(f"✅ Ollama 服務已就緒（等待了 {elapsed} 秒）")
            break
    except requests.exceptions.ConnectionError:
        # 服務尚未啟動，繼續等待
        pass
    time.sleep(1)
else:
    raise RuntimeError(
        "⏰ Ollama 啟動超時（30秒）！\n"
        "請嘗試：Runtime → Restart session，然後重新執行此 Cell"
    )

# ── 驗證 GPU 是否可用 ────────────────────────────────────────
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                           "--format=csv,noheader"],
                          capture_output=True, text=True)
if gpu_info.returncode == 0:
    print(f"🎮 偵測到 GPU：{gpu_info.stdout.strip()}")
else:
    print("⚠️  未偵測到 GPU，將使用 CPU 推論（速度較慢）")
    print("   建議：Runtime → Change runtime type → T4 GPU")

In [ ]:
# ============================================================
# 🔽 Cell 5｜下載 AI 模型（約 5~15 分鐘）
# ============================================================

print("🔽 步驟 3/5：下載 AI 模型")
print(f"   📐 嵌入模型：{EMBEDDING_MODEL}（約 670MB）")
print(f"   💬 對話模型：{LLM_MODEL}（約 4.1GB）")
print("   ⚠️  首次執行約需 5~15 分鐘，請耐心等候...\n")

# ── 下載嵌入模型 ────────────────────────────────────────────
print(f"⏬ [1/2] 下載嵌入模型 {EMBEDDING_MODEL}...")
!ollama pull {EMBEDDING_MODEL}
print(f"✅ {EMBEDDING_MODEL} 下載完成\n")

# ── 下載對話 LLM 模型 ────────────────────────────────────────
print(f"⏬ [2/2] 下載 LLM 模型 {LLM_MODEL}...")
!ollama pull {LLM_MODEL}
print(f"✅ {LLM_MODEL} 下載完成\n")

# ── 列出已安裝的模型 ─────────────────────────────────────────
print("📋 目前已安裝的模型：")
!ollama list

In [ ]:
# ============================================================
# 📦 Cell 6｜安裝 Python 套件
# ============================================================

print("📦 步驟 4/5：安裝 Python 套件...")

# 安裝所有必要的 Python 套件
# streamlit  : Web UI 框架
# pyngrok    : Ngrok Python 包裝，用於建立公開 HTTPS 隧道
# chromadb   : 向量資料庫
# pandas     : 資料處理（讀取 Excel）
# openpyxl   : pandas 讀取 .xlsx 格式的依賴
# ollama     : Ollama Python SDK
!pip install -q streamlit pyngrok chromadb pandas openpyxl ollama

print("✅ Python 套件安裝完成")

# 顯示關鍵套件版本
import importlib
for pkg in ["streamlit", "chromadb", "ollama", "pyngrok"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"   {pkg}: {mod.__version__}")
    except (ImportError, AttributeError):
        print(f"   {pkg}: 已安裝（版本未知）")

In [ ]:
# ============================================================
# 📊 Cell 7｜準備知識庫資料（QA Excel 檔案）
# ============================================================

import pandas as pd
import os

print("📊 準備知識庫資料...")

# ── 選配：掛載 Google Drive ──────────────────────────────────
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    print("💾 掛載 Google Drive...")
    drive.mount("/content/drive", force_remount=False)

    # 建立所需目錄（若不存在）
    os.makedirs(os.path.dirname(GDRIVE_QA_PATH), exist_ok=True)
    os.makedirs(CHROMA_PERSIST_DIR, exist_ok=True)

    # 實際使用的 QA 檔案路徑
    qa_file_path = GDRIVE_QA_PATH
    chroma_path  = CHROMA_PERSIST_DIR
    print(f"✅ Google Drive 已掛載")
    print(f"   📁 QA 檔案路徑   : {qa_file_path}")
    print(f"   📁 向量庫路徑    : {chroma_path}")
else:
    # 記憶體模式：使用 Colab 本地臨時路徑
    qa_file_path = "/content/QA50.xlsx"
    chroma_path  = ""
    print("ℹ️  使用記憶體模式（不掛載 Drive）")

print()

# ── 若 QA 檔案不存在，自動建立範例知識庫 ──────────────────
if not os.path.exists(qa_file_path):
    print(f"⚠️  未找到 QA 檔案，自動建立範例知識庫：{qa_file_path}")
    print("   提示：您也可以將自己的 QA50.xlsx 上傳至上述路徑以取代範例資料\n")

    # 範例繁體中文 QA 知識庫（10 筆，涵蓋本教學相關知識）
    sample_qa = [
        "RAG 是什麼？RAG（Retrieval-Augmented Generation，檢索增強生成）是一種結合資訊檢索與生成式 AI 的技術，能讓 LLM 根據外部知識庫動態地回答問題，有效降低幻覺（Hallucination）現象。",
        "Ollama 是什麼？Ollama 是一個開源工具，讓您可以在本地端輕鬆下載並執行各種大型語言模型（LLM），無需依賴付費雲端 API，保護資料隱私。",
        "ChromaDB 是什麼？ChromaDB 是一個開源的向量資料庫，專門用來儲存和查詢文字嵌入向量，常用於 RAG 應用開發，支援持久化儲存與記憶體模式。",
        "Streamlit 是什麼？Streamlit 是一個 Python 函式庫，能讓開發者以極少的程式碼快速建立互動式的機器學習和數據應用網頁介面，非常適合原型開發。",
        "什麼是嵌入向量（Embedding）？嵌入向量是將文字轉換為高維度數字向量的表示方式，相似語意的文字會有相近的向量。向量搜尋就是利用這個特性找出語意最接近的文件。",
        "Breeze-7B 是什麼？Breeze-7B 是聯發科技（MediaTek Research）基於 Mistral-7B 進行繁體中文持續訓練的語言模型，在繁體中文理解和生成能力上有顯著提升，適合台灣在地應用。",
        "mxbai-embed-large 是什麼？mxbai-embed-large 是 mixedbread.ai 開發的高效能文字嵌入模型，在 MTEB（Massive Text Embedding Benchmark）多項語義搜尋基準測試中表現優異。",
        "向量搜尋如何運作？向量搜尋首先將查詢問題轉為嵌入向量，再在向量資料庫中計算與所有文件向量的餘弦相似度，返回相似度最高的 Top-K 個文件作為參考知識。",
        "什麼是 Streamlit Session State？在 Streamlit 中，st.session_state 是一個字典物件，用於在使用者互動之間（如按下按鈕、輸入文字）持久保存變數狀態，避免每次頁面重新渲染都重新執行初始化程式碼。",
        "Google Colab 免費版有什麼限制？免費版每次 Session 最長約 12 小時，使用 GPU 時有每日配額限制，Runtime 閒置一段時間後會自動斷線，且重新連線後需重新安裝套件和下載模型。建議搭配 Google Drive 持久化重要資料。",
    ]

    # 建立只有一欄的 DataFrame（符合原始程式讀取格式，無標題列）
    df_sample = pd.DataFrame(sample_qa)
    df_sample.to_excel(qa_file_path, index=False, header=False)
    print(f"✅ 已建立含 {len(sample_qa)} 筆 QA 的範例知識庫")

# ── 讀取並顯示知識庫預覽 ────────────────────────────────────
df = pd.read_excel(qa_file_path, header=None)
print(f"\n📋 知識庫已載入：共 {len(df)} 筆資料")
print("\n前 3 筆預覽：")
for i, row in df.head(3).iterrows():
    preview = str(row[0])[:80]
    suffix  = "..." if len(str(row[0])) > 80 else ""
    print(f"  [{i:02d}] {preview}{suffix}")

In [ ]:
# ============================================================
# ✍️ Cell 8｜產生 Streamlit 應用程式檔案 (app.py)
# ============================================================

# 注意：%%writefile 魔法指令必須是 Cell 的第一行
# 以下程式碼將被寫入 /content/app.py
print("✍️ 正在產生 app.py...")

app_code = '''
# -*- coding: utf-8 -*-
"""
LLM + RAG 本地知識問答系統
改寫自 YouTube 教學，適配 Google Colab 部署

技術棧：
  - Streamlit     : Web UI 框架
  - Ollama        : 本地 LLM 推論引擎
  - ChromaDB      : 向量資料庫（語義搜尋）
  - mxbai-embed-large        : 文字嵌入模型
  - ycchen/breeze-7b-instruct-v1_0 : 繁體中文 LLM
"""

import streamlit as st      # Web UI 框架
import ollama               # Ollama Python SDK
import chromadb             # 向量資料庫
import pandas as pd         # 資料處理
import os                   # 系統路徑操作


# ── 常數設定（由 Colab 啟動指令透過環境變數傳入）────────────

# 文字嵌入模型名稱
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "mxbai-embed-large")

# 對話語言模型名稱
LLM_MODEL = os.environ.get("LLM_MODEL", "ycchen/breeze-7b-instruct-v1_0")

# 知識庫 Excel 檔案路徑
QA_FILE_PATH = os.environ.get("QA_FILE_PATH", "/content/QA50.xlsx")

# ChromaDB 持久化路徑（空字串 = 使用記憶體模式）
CHROMA_PERSIST = os.environ.get("CHROMA_PERSIST", "")


def get_chromadb_client():
    """
    根據是否設定持久化路徑，建立並回傳 ChromaDB 客戶端。

    - 有持久化路徑：PersistentClient（資料寫入磁碟，支援重啟後快速還原）
    - 無持久化路徑：EphemeralClient（資料存於記憶體，每次重啟清空）

    回傳：
        chromadb.ClientAPI：已初始化的 ChromaDB 客戶端實例
    """
    if CHROMA_PERSIST:
        # 持久化模式：資料寫入 Google Drive
        return chromadb.PersistentClient(path=CHROMA_PERSIST)
    else:
        # 記憶體模式：每次重啟需重新建立索引
        return chromadb.EphemeralClient()


def setup_database():
    """
    初始化向量資料庫，執行步驟如下：
      1. 讀取 Excel 知識庫檔案（無標題列，單一文字欄）
      2. 對每筆資料呼叫 Ollama 嵌入 API 產生語義向量
      3. 將文字與向量一同存入 ChromaDB 集合

    執行完畢後：
      - 將集合物件存入 st.session_state.collection
      - 設定 st.session_state.already_executed = True
    """
    # 建立 ChromaDB 客戶端（持久化或記憶體模式）
    client = get_chromadb_client()

    # 讀取 Excel 知識庫（無標題列，第 0 欄為文字內容）
    documents = pd.read_excel(QA_FILE_PATH, header=None)
    total_docs = len(documents)

    # 取得或建立名為 knowledge_base 的向量集合
    # hnsw:space=cosine 表示使用餘弦相似度計算距離
    collection = client.get_or_create_collection(
        name="knowledge_base",
        metadata={"hnsw:space": "cosine"}
    )

    # 顯示建立索引的進度條
    progress_bar = st.progress(0, text="🔄 正在建立知識庫向量索引...")

    # 逐筆對文件產生嵌入向量並寫入集合
    for index, content in documents.iterrows():
        # 呼叫 Ollama 嵌入 API，將文字轉為語義向量
        response = ollama.embeddings(
            model=EMBEDDING_MODEL,
            prompt=str(content[0])
        )

        # 將文字、向量與唯一 ID 存入 ChromaDB
        collection.add(
            ids=[str(index)],
            embeddings=[response["embedding"]],
            documents=[str(content[0])]
        )

        # 更新進度條（百分比 + 目前/總計筆數）
        progress_pct = (index + 1) / total_docs
        progress_bar.progress(
            progress_pct,
            text=f"🔄 向量索引建立中... ({index + 1}/{total_docs})"
        )

    progress_bar.progress(1.0, text=f"✅ 知識庫索引建立完成！（共 {total_docs} 筆）")

    # 將集合存入 session_state 供查詢函式使用
    st.session_state.collection      = collection
    # 標記已完成初始化，下次渲染不會重複執行
    st.session_state.already_executed = True


def initialize():
    """
    檢查 Session State 並按需初始化向量資料庫。
    此函式於每次 Streamlit 頁面渲染時呼叫，
    但實際的 setup_database() 只在首次執行時觸發。
    """
    # 若 session_state 中不存在此 key，表示全新 Session
    if "already_executed" not in st.session_state:
        st.session_state.already_executed = False

    # 尚未完成初始化時，執行資料庫設置
    if not st.session_state.already_executed:
        with st.spinner("⏳ 首次啟動：正在建立知識庫索引，請稍候..."):
            setup_database()


def handle_user_input(user_input: str, collection) -> None:
    """
    處理使用者問題的完整 RAG 流程：
      1. 將問題轉為嵌入向量
      2. 在向量資料庫中查詢 Top-3 最相關知識
      3. 組合 Prompt 並呼叫 LLM 生成繁體中文回答
      4. 在頁面顯示回答與參考來源

    參數：
        user_input (str) : 使用者輸入的問題文字
        collection       : ChromaDB 集合物件（語義搜尋用）
    """
    with st.spinner("🤔 LLM 思考中，請稍候（約 10~30 秒）..."):

        # ── Step 1：將問題轉為嵌入向量 ─────────────────────────
        query_response = ollama.embeddings(
            prompt=user_input,
            model=EMBEDDING_MODEL
        )

        # ── Step 2：語義搜尋，取 Top-3 最相關知識片段 ──────────
        results = collection.query(
            query_embeddings=[query_response["embedding"]],
            n_results=3
        )
        relevant_docs = results["documents"][0]  # 最相關的文件清單
        context_text  = "\n".join(relevant_docs) # 拼接為上下文字串

        # ── Step 3：組合 Prompt 並呼叫 LLM 生成回答 ────────────
        prompt = (
            f"以下是從知識庫中檢索到的相關內容：\n"
            f"{context_text}\n\n"
            f"請根據上述知識內容，用繁體中文詳細回答以下問題：\n"
            f"{user_input}"
        )
        llm_output = ollama.generate(
            model=LLM_MODEL,
            prompt=prompt
        )

    # ── Step 4：顯示回答與參考來源 ──────────────────────────
    st.markdown("### 💬 回答")
    st.write(llm_output["response"])

    # 可展開的參考來源區塊
    with st.expander("📚 參考的知識片段（點擊展開）", expanded=False):
        for idx, doc in enumerate(relevant_docs, 1):
            st.markdown(f"**[{idx}]** {doc}")
            if idx < len(relevant_docs):
                st.divider()


def main():
    """
    Streamlit 應用程式主函式：
      - 設定頁面基本屬性（標題、圖示、版面）
      - 呼叫初始化（首次執行建立知識庫索引）
      - 渲染問答輸入介面與操作按鈕
      - 處理使用者送出事件
    """
    # ── 頁面基本設定 ─────────────────────────────────────────
    st.set_page_config(
        page_title="LLM + RAG 知識問答",
        page_icon="🤖",
        layout="centered",
        initial_sidebar_state="collapsed"
    )

    # ── 初始化（只有首次執行才真正觸發 setup_database）──────
    initialize()

    # ── 頁面標題與模型資訊 ───────────────────────────────────
    st.title("🤖 我的第一個 LLM + RAG 本地知識問答")
    st.markdown(
        f"> 嵌入：`{EMBEDDING_MODEL}` ｜ 生成：`{LLM_MODEL}`  \n"
        f"> 知識庫：`{QA_FILE_PATH}`"
    )
    st.divider()

    # ── 問題輸入區 ───────────────────────────────────────────
    user_input = st.text_area(
        "💭 您想問什麼？",
        placeholder="例如：RAG 是什麼？Breeze-7B 有什麼特色？",
        height=120
    )

    # ── 操作按鈕（送出 / 清除）──────────────────────────────
    col_submit, col_clear, _ = st.columns([1, 1, 3])
    with col_submit:
        submit_btn = st.button("🚀 送出", type="primary", use_container_width=True)
    with col_clear:
        clear_btn  = st.button("🗑️ 清除", use_container_width=True)

    # ── 事件處理 ─────────────────────────────────────────────
    if submit_btn:
        if user_input.strip():
            # 執行完整 RAG 問答流程
            handle_user_input(user_input, st.session_state.collection)
        else:
            # 使用者未輸入問題時顯示警告
            st.warning("⚠️ 請先輸入您想詢問的問題！")

    if clear_btn:
        # 重新渲染頁面（清空輸出，但保留已初始化的知識庫）
        st.rerun()


if __name__ == "__main__":
    main()
'''

# 將 app.py 寫入 Colab 環境
with open("/content/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print(f"✅ app.py 產生完成（{len(app_code)} bytes）")
print("   路徑：/content/app.py")

In [ ]:
# ============================================================
# 🚀 Cell 9｜啟動 Streamlit + Ngrok 公開隧道（最後一步）
# ============================================================

import subprocess
import time
import os
from pyngrok import ngrok, conf

print("🚀 步驟 5/5：啟動應用程式...\n")

# ── 傳遞設定給 Streamlit（透過環境變數）────────────────────
# 這樣 app.py 啟動時就能讀到正確的模型名稱和路徑
os.environ["EMBEDDING_MODEL"] = EMBEDDING_MODEL
os.environ["LLM_MODEL"]       = LLM_MODEL
os.environ["QA_FILE_PATH"]    = qa_file_path
os.environ["CHROMA_PERSIST"]  = chroma_path

# ── 設定 Ngrok 認證 token ────────────────────────────────────
conf.get_default().auth_token = NGROK_AUTHTOKEN
print("✅ Ngrok token 設定完成")

# ── 確保沒有殘留的 Streamlit 程序 ─────────────────────────
!pkill -f streamlit 2>/dev/null; echo "舊程序已清理"
time.sleep(2)

# ── 在背景啟動 Streamlit 服務 ────────────────────────────────
# --server.headless=true        : 無瀏覽器模式（Colab 必要）
# --server.fileWatcherType=none : 關閉檔案監視器（Colab 環境穩定性）
# --server.port=8501            : 固定使用 8501 埠
streamlit_proc = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.fileWatcherType=none",
        "--browser.gatherUsageStats=false",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ 等待 Streamlit 啟動（5 秒）...")
time.sleep(5)

# ── 確認 Streamlit 程序仍在執行 ────────────────────────────
if streamlit_proc.poll() is not None:
    # 程序已結束，表示啟動失敗
    stdout, stderr = streamlit_proc.communicate()
    raise RuntimeError(
        f"❌ Streamlit 啟動失敗！\n"
        f"錯誤訊息：{stderr.decode('utf-8', errors='replace')}"
    )

print("✅ Streamlit 服務已啟動（port 8501）")

# ── 建立 Ngrok 公開隧道 ──────────────────────────────────────
# 將 Colab 內部的 localhost:8501 映射為對外的 HTTPS URL
print("🌐 建立 Ngrok 公開隧道...")
tunnel = ngrok.connect(8501)
public_url = tunnel.public_url

# ── 顯示結果 ─────────────────────────────────────────────────
print()
print("=" * 62)
print(f"  🌐 應用程式公開網址：")
print(f"  👉 {public_url}")
print("=" * 62)
print()
print("📌 注意事項：")
print("  • 此連結在本次 Colab Session 期間有效")
print("  • Runtime 斷線後連結失效，需重新執行 Cell 4 → Cell 9")
print("  • 若需停止：執行 ngrok.kill() 並重啟 Runtime")
print()
print("🎉 首次開啟連結時，Streamlit 需要幾秒鐘載入，請稍候！")